# Darcy-CNN Profiling

In [1]:
# cd("/content")
# isdir("/content/darcy-cnn")
# rm("/content/darcy-cnn", recursive=true, force=true)
# run(`git clone https://github.com/thezettascale/darcy-cnn.git /content/darcy-cnn`)
# cd("/content/darcy-cnn")
# @assert isfile("Project.toml")
# println("Fresh clone ready at ", pwd())

# # Cache packages to Google Drive if mounted
# drive_depot = joinpath(ENV["HOME"], "drive", "MyDrive", "julia_depot")
# if isdir(dirname(drive_depot))
#     mkpath(drive_depot)
#     pushfirst!(DEPOT_PATH, drive_depot)
#     @info "Package cache: $drive_depot"
# end

# using Pkg; Pkg.activate(pwd()); Pkg.resolve(); Pkg.instantiate()

In [2]:
ENV["DEVICE"] = "gpu"

using Random, Lux, Lux.Training, Reactant, Optimisers, Statistics
using Printf, JSON
using MLDataDevices: reactant_device, cpu_device

cd("..")
include("src/WavKANConv.jl"); using .WavKANConv

In [3]:
MODEL_NAMES = ["CNN", "FNO", "KAN_CNN"]
PROFILE_BS = 2

function setup(name::String; dev=reactant_device())
    rng = Random.default_rng()
    Random.seed!(rng, 42)

    cfg = load_config(name)
    train_loader, test_loader = get_darcy_loader(PROFILE_BS; dev=dev)
    x_sample, y_sample = first(train_loader)

    model = create_model(cfg)
    ps, st = Lux.setup(rng, model)
    ps, st = ps |> dev, st |> dev
    st_test = Lux.testmode(st)
    n_params = Lux.parameterlength(model)

    loss_fn(y_pred, y) = loss_fcn(y_pred, y; p=cfg.p)
    function objective(model, ps, st, (x, y))
        y_pred, st_new = model(x, ps, st)
        return loss_fn(y_pred, y), st_new, (;)
    end
    ts = Training.TrainState(model, ps, st, Optimisers.Adam(cfg.learning_rate))

    println("$name: $n_params params")
    return (; model, ps, st, st_test, cfg, n_params, ts, objective, x_sample, y_sample)
end

function hlo_op_profile(hlo_str::String)
    ops = String[]
    for line in split(hlo_str, '\n')
        m = match(r"stablehlo\.(\w+)", line)
        m !== nothing && push!(ops, m.captures[1])
    end
    total = length(ops)
    d = Dict{String,Int}()
    for x in ops; d[x] = get(d, x, 0) + 1; end
    counts = sort(collect(d); by=last, rev=true)
    return [(op, n, 100.0 * n / total) for (op, n) in counts]
end

function print_hlo_profile(name, profile)
    total = sum(x[2] for x in profile)
    @printf("\n=== %s (%d total ops) ===\n", name, total)
    @printf("%-25s %8s %8s\n", "Op", "Count", "%")
    println("-"^43)
    for (op, n, pct) in profile
        @printf("%-25s %8d %7.1f%%\n", op, n, pct)
    end
end

function profile_and_bench(name, f, args...; warmup=3, trials=10)
    hlo = string(Reactant.@code_hlo f(args...))
    profile = hlo_op_profile(hlo)
    print_hlo_profile(name, profile)

    compiled = Reactant.@compile sync=true f(args...)
    for _ in 1:warmup; compiled(args...); end
    times = [(@elapsed compiled(args...)) * 1000 for _ in 1:trials]
    @printf("%-20s  med=%.3f ms  min=%.3f ms  max=%.3f ms\n\n", name, median(times), minimum(times), maximum(times))

    return profile, times, compiled
end

profile_and_bench (generic function with 1 method)

## Profile

In [4]:
device = ENV["DEVICE"]
all_results = Dict{String, Any}()

for name in MODEL_NAMES
    m = setup(name)

    # Profile forward pass (HLO + timing) — testmode, no BatchNorm training constraint
    fwd(x, ps, st) = m.model(x, ps, st)
    fwd_profile, fwd_times, _ = profile_and_bench("$(name)_forward", fwd, m.x_sample, m.ps, m.st_test)

    # Benchmark train step (single_train_step! compiles & caches internally, use sync=true for timing)
    Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), m.ts; sync=true)  # warmup/compile
    ts = m.ts
    ts_times = Float64[]
    for _ in 1:10
        t = @elapsed begin
            _, _, _, ts = Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), ts; sync=true)
        end
        push!(ts_times, t * 1000)
    end
    @printf("%-20s  med=%.3f ms  min=%.3f ms  max=%.3f ms\n\n",
            "$(name)_train_step", median(ts_times), minimum(ts_times), maximum(ts_times))

    all_results[name] = Dict(
        "n_params" => m.n_params,
        "forward" => Dict(
            "timing_ms" => Dict("median" => median(fwd_times), "min" => minimum(fwd_times), "max" => maximum(fwd_times)),
            "hlo_ops" => [Dict("op" => op, "count" => n, "pct" => round(pct; digits=1)) for (op, n, pct) in fwd_profile],
            "total_ops" => sum(x[2] for x in fwd_profile),
        ),
        "train_step" => Dict(
            "timing_ms" => Dict("median" => median(ts_times), "min" => minimum(ts_times), "max" => maximum(ts_times)),
        ),
    )
end

# Save
mkpath("profiles")
outfile = "profiles/profile_$(device).json"
open(outfile, "w") do io
    JSON.print(io, Dict("device" => device, "models" => all_results), 2)
end
println("Saved to $outfile")

CNN: 5982121 params


I0000 00:00:1775953783.298575  135101 service.cc:153] XLA service 0x2bfd9e10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775953783.298590  135101 service.cc:161]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
I0000 00:00:1775953783.298872  135101 se_gpu_pjrt_client.cc:1440] Using BFC allocator.
I0000 00:00:1775953783.298895  135101 gpu_helpers.cc:141] XLA backend allocating 2923905024 bytes on device 0 for BFCAllocator.
I0000 00:00:1775953783.298912  135101 gpu_helpers.cc:182] XLA backend will use up to 974635008 bytes on device 0 for CollectiveBFCAllocator.
I0000 00:00:1775953783.309457  135101 cuda_dnn.cc:461] Loaded cuDNN version 91400



=== CNN_forward (55 total ops) ===
Op                           Count        %
-------------------------------------------
constant                        10    18.2%
add                              7    12.7%


W0000 00:00:1775953818.600061  135183 hlo_rematerialization.cc:3204] Can't reduce memory use below 2.75GiB (2951519032 bytes) by rematerialization; only reduced to 8.19GiB (8798446112 bytes), down from 8.19GiB (8798446112 bytes) originally
W0000 00:00:1775953818.600235  135182 hlo_rematerialization.cc:3204] Can't reduce memory use below 2.75GiB (2951519032 bytes) by rematerialization; only reduced to 8.19GiB (8798446112 bytes), down from 8.19GiB (8798446112 bytes) originally
W0000 00:00:1775953829.031845  135101 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 8.19GiB (rounded to 8792548352)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1775953829.031956  135101 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1775953829.031965  135101 bfc_allocator.cc

convolution                      7    12.7%
compare                          7    12.7%
select                           7    12.7%
broadcast_in_dim                 7    12.7%
multiply                         7    12.7%
reverse                          3     5.5%
CNN_forward           med=9.449 ms  min=9.327 ms  max=9.549 ms

CNN_train_step        med=37.035 ms  min=36.001 ms  max=2910.284 ms

FNO: 4667665 params

=== FNO_forward (244 total ops) ===
Op                           Count        %
-------------------------------------------
multiply                        68    27.9%
add                             39    16.0%
broadcast_in_dim                31    12.7%
constant                        19     7.8%
dynamic_update_slice            16     6.6%
convolution                     14     5.7%
pad                             12     4.9%
tanh                            10     4.1%
slice                            8     3.3%
fft                              8     3.3%
reduce            

E0000 00:00:1775954740.530978  135101 cuda_fft.cc:256] Failed to make cuFFT batched plan: 5
E0000 00:00:1775954740.531007  135101 cuda_fft.cc:323] Initialize Params: rank: 2 elem_count: 32 input_embed: 32 input_stride: 1 input_distance: 1024 output_embed: 32 output_stride: 1 output_distance: 544 batch_count: 188
E0000 00:00:1775954740.534523  135101 cuda_fft.cc:332] Failed to initialize batched cufft plan with customized allocator: Failed to make cuFFT batched plan.
E0000 00:00:1775954740.598520  135101 status_macros.cc:58] INTERNAL: RET_CHECK failure (external/xla/xla/backends/gpu/runtime/fft_thunk.cc:199) fft_plan != nullptr Failed to create cuFFT batched plan with scratch allocator
*** Begin stack trace ***
	tsl::CurrentStackTrace[abi:cxx11]()
	
	xla::status_macros::MakeErrorStream::Impl::GetStatus()
	xla::gpu::RunFft(stream_executor::DeviceAddressBase, xla::Shape const&, stream_executor::DeviceAddressBase, xla::Shape const&, stream_executor::fft::Type, absl::lts_20250814::Span<long

LoadError: INTERNAL: RET_CHECK failure (external/xla/xla/backends/gpu/runtime/fft_thunk.cc:199) fft_plan != nullptr Failed to create cuFFT batched plan with scratch allocator


## XLA Trace
Upload `.pb` to [Perfetto UI](https://ui.perfetto.dev) for per-op runtime breakdown.

In [5]:
device = ENV["DEVICE"]
for name in MODEL_NAMES
    m = setup(name)

    # Warmup (compiles & caches internally)
    Training.single_train_step!(AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), m.ts; sync=true)

    tracefile = "profiles/$(name)_train_step_$(device)"
    Reactant.Profiler.with_profiler(tracefile) do
        ts = m.ts
        for _ in 1:20
            _, _, _, ts = Training.single_train_step!(
                AutoEnzyme(), m.objective, (m.x_sample, m.y_sample), ts; sync=true
            )
        end
    end
    println("$name -> $tracefile/")
end

CNN: 5982121 params
CNN -> profiles/CNN_train_step_gpu/


I0000 00:00:1775954744.092932  135101 profiler_session.cc:103] Profiler session initializing.
I0000 00:00:1775954744.092948  135101 profiler_session.cc:118] Profiler session started.
I0000 00:00:1775954744.092967  135101 cupti_tracer.cc:1104] Profiler found 1 GPUs
I0000 00:00:1775954744.101654  135101 cuda_version_variants.cc:36] CUDA runtime version: 13020
I0000 00:00:1775954744.101675  135101 cuda_version_variants.cc:52] CUDA driver version: 13020
E0000 00:00:1775954744.114665  135101 cupti_error_manager.cc:226] cuptiSubscribe: error 15: CUPTI_ERROR_NOT_INITIALIZED
E0000 00:00:1775954744.114685  135101 cupti_error_manager.cc:255] cuptiGetResultString: ignored due to a previous error.
E0000 00:00:1775954744.114687  135101 cupti_tracer.cc:1375] function Subscribe(&subscriber_, (CUpti_CallbackFunc)ApiCallback, this)failed with error 
E0000 00:00:1775954744.114692  135101 profiler_controller.cc:51] INTERNAL: CUPTI call error: 
I0000 00:00:1775954747.287964  135101 profiler_session.cc:68]

FNO: 4667665 params


I0000 00:00:1775954878.242558  135101 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775954878.252751  135101 dot_merger.cc:476] Merging Dots in computation: main.36
I0000 00:00:1775954878.260897  135101 dot_merger.cc:476] Merging Dots in computation: main.36
E0000 00:00:1775954906.504214  135101 cuda_fft.cc:256] Failed to make cuFFT batched plan: 5
E0000 00:00:1775954906.504264  135101 cuda_fft.cc:323] Initialize Params: rank: 2 elem_count: 32 input_embed: 32 input_stride: 1 input_distance: 1024 output_embed: 32 output_stride: 1 output_distance: 544 batch_count: 188
E0000 00:00:1775954906.506319  135101 cuda_fft.cc:332] Failed to initialize batched cufft plan with customized allocator: Failed to make cuFFT batched plan.
E0000 00:00:1775954906.576491  135101 status_macros.cc:58] INTERNAL: RET_CHECK failure (external/xla/xla/backends/gpu/runtime/fft_thunk.cc:199) fft_plan != nullptr Failed to create cuFFT batched plan with scratch allocator
*** Begin stack trace **

LoadError: INTERNAL: RET_CHECK failure (external/xla/xla/backends/gpu/runtime/fft_thunk.cc:199) fft_plan != nullptr Failed to create cuFFT batched plan with scratch allocator
